# Notebook 06 — LTR Dataset Validation

Validation-only notebook. Does NOT create new features or perform any splits.

Runs 6 checks before model training:
- **Check A**: Required columns present
- **Check B**: Target distribution
- **Check C**: Group structure
- **Check D**: Task leakage
- **Check E**: Employee cold-start coverage
- **Check F**: Ranking group arrays

## Setup — Load LTR Datasets

In [1]:
import pandas as pd
import numpy as np
import json
import os
import datetime
import warnings
warnings.filterwarnings("ignore")

LTR_DIR = "../data/processed/ltr_datasets"

ltr_train = pd.read_csv(f"{LTR_DIR}/ltr_train_dataset.csv")
ltr_val   = pd.read_csv(f"{LTR_DIR}/ltr_validation_dataset.csv")
ltr_test  = pd.read_csv(f"{LTR_DIR}/ltr_test_dataset.csv")

with open(f"{LTR_DIR}/ltr_feature_columns.json") as f:
    ltr_meta = json.load(f)

FEATURE_COLS = ltr_meta["feature_cols"]

print(f"Train: {ltr_train.shape}, Val: {ltr_val.shape}, Test: {ltr_test.shape}")
print(f"Feature columns from metadata: {len(FEATURE_COLS)}")

Train: (36605, 65), Val: (7829, 65), Test: (7827, 65)
Feature columns from metadata: 56


## Check A — Required Columns Present

In [2]:
REQUIRED_COLS = [
    "Task_ID", "Employee_ID", "relevance",
    "Estimated_Planned_Hours", "Planned_Hours_Log",
    "Task_Text_Length", "Task_Word_Count",
    "Has_Task_Description", "Has_Deadline", "Days_To_Deadline",
    "Task_Skill_Count",
    "employee_historical_task_count",
    "employee_historical_avg_planned_hours",
    "employee_historical_avg_skill_count",
    "employee_historical_unique_projects",
    "employee_task_skill_match_count",
    "employee_task_skill_match_ratio",
    "employee_has_matching_skill",
    "employee_project_task_count",
    "employee_has_project_experience",
]

missing_cols = [c for c in REQUIRED_COLS if c not in ltr_train.columns]
if missing_cols:
    print(f"FAIL — Missing required columns: {missing_cols}")
else:
    print("PASS — All required columns present in ltr_train")

print(f"\nTotal columns: {len(ltr_train.columns)}")
print(f"Feature columns: {len(FEATURE_COLS)}")
print("\nAll columns and dtypes:")
for c in ltr_train.columns:
    print(f"  {c}: {ltr_train[c].dtype}")

PASS — All required columns present in ltr_train

Total columns: 65
Feature columns: 56

All columns and dtypes:
  Task_ID: str
  Employee_ID: str
  relevance: int64
  Hours_Spent: float64
  FLAG_LEAKAGE_Timesheet_Work_Logs: str
  Task_Priority: str
  Estimated_Planned_Hours: float64
  FLAG_LEAKAGE_Actual_Hours_Spent: float64
  FLAG_LEAKAGE_Timesheet_Logs_Count: int64
  FLAG_LEAKAGE_Task_Stage: str
  FLAG_LEAKAGE_All_Collaborating_Employees: str
  Task_Text_Length: int64
  Task_Word_Count: int64
  Task_Description_Length: int64
  Task_Name_Length: int64
  Has_Task_Description: int64
  Created_Year: int64
  Created_Month: int64
  Created_DayOfWeek: int64
  Created_Quarter: int64
  Days_To_Deadline: float64
  Has_Deadline: int64
  Planned_Hours_Log: float64
  Planned_Task_Size: str
  Task_Skill_Count: int64
  Skill_Odoo_ERP_Development: int64
  Skill_Database_Management: int64
  Skill_Server_Administration: int64
  Skill_Project_Management: int64
  Skill_Software_Testing: int64
  Skill_W

## Check B — Target Distribution

In [3]:
print("TARGET DISTRIBUTION PER SPLIT:")
print("=" * 55)
for name, df in [("train", ltr_train), ("val", ltr_val), ("test", ltr_test)]:
    n_pos   = (df["relevance"] == 1).sum()
    n_neg   = (df["relevance"] == 0).sum()
    ratio   = n_pos / n_neg if n_neg > 0 else float("inf")
    print(f"{name}:")
    print(f"  Positive examples:          {n_pos:,}")
    print(f"  Negative examples:          {n_neg:,}")
    print(f"  Positive-to-negative ratio: {ratio:.4f}")
    print(f"  Total pairs:                {len(df):,}")
    pos_per_task = df[df["relevance"]==1].groupby("Task_ID").size()
    print(f"  Avg positives per task:     {pos_per_task.mean():.2f}")
    print(f"  Min positives per task:     {pos_per_task.min()}")
    print(f"  Max positives per task:     {pos_per_task.max()}")
    print()
print("Do NOT blindly oversample based on class imbalance alone.")
print("For ranking, evaluate positive/negative balance per task group.")

TARGET DISTRIBUTION PER SPLIT:
train:
  Positive examples:          2,425
  Negative examples:          34,180
  Positive-to-negative ratio: 0.0709
  Total pairs:                36,605
  Avg positives per task:     1.42
  Min positives per task:     1
  Max positives per task:     29

val:
  Positive examples:          509
  Negative examples:          7,320
  Positive-to-negative ratio: 0.0695
  Total pairs:                7,829
  Avg positives per task:     1.39
  Min positives per task:     1
  Max positives per task:     24

test:
  Positive examples:          487
  Negative examples:          7,340
  Positive-to-negative ratio: 0.0663
  Total pairs:                7,827
  Avg positives per task:     1.33
  Min positives per task:     1
  Max positives per task:     14

Do NOT blindly oversample based on class imbalance alone.
For ranking, evaluate positive/negative balance per task group.


## Check C — Group Structure

In [4]:
print("GROUP STRUCTURE VERIFICATION:")
print("=" * 55)
for name, df in [("train", ltr_train), ("val", ltr_val), ("test", ltr_test)]:
    group_check = df.groupby("Task_ID")["relevance"].agg(["min", "max", "count"])
    tasks_no_pos = (group_check["max"] < 1).sum()
    tasks_no_neg = (group_check["min"] > 0).sum()
    if tasks_no_pos > 0:
        print(f"FAIL — {name}: {tasks_no_pos} tasks have NO positive employee!")
    if tasks_no_neg > 0:
        print(f"FAIL — {name}: {tasks_no_neg} tasks have NO negative employee!")
    n_positives_per_task = df[df["relevance"]==1].groupby("Task_ID").size()
    group_size = df.groupby("Task_ID").size()
    print(f"{name}:")
    print(f"  Total tasks:                    {df["Task_ID"].nunique()}")
    print(f"  Tasks with exactly 1 positive:  {(n_positives_per_task == 1).sum()}")
    print(f"  Tasks with multiple positives:  {(n_positives_per_task > 1).sum()}")
    print(f"  Average candidates per task:    {group_size.mean():.1f}")
    print(f"  Min candidates per task:        {group_size.min()}")
    print(f"  Max candidates per task:        {group_size.max()}")
    if tasks_no_pos == 0 and tasks_no_neg == 0:
        print(f"  PASS — All tasks have at least 1 positive and 1 negative.")
    print()

GROUP STRUCTURE VERIFICATION:
train:
  Total tasks:                    1709
  Tasks with exactly 1 positive:  1394
  Tasks with multiple positives:  315
  Average candidates per task:    21.4
  Min candidates per task:        21
  Max candidates per task:        49
  PASS — All tasks have at least 1 positive and 1 negative.

val:
  Total tasks:                    366
  Tasks with exactly 1 positive:  295
  Tasks with multiple positives:  71
  Average candidates per task:    21.4
  Min candidates per task:        21
  Max candidates per task:        44
  PASS — All tasks have at least 1 positive and 1 negative.

test:
  Total tasks:                    367
  Tasks with exactly 1 positive:  311
  Tasks with multiple positives:  56
  Average candidates per task:    21.3
  Min candidates per task:        21
  Max candidates per task:        34
  PASS — All tasks have at least 1 positive and 1 negative.



## Check D — Task Leakage

In [5]:
train_tasks = set(ltr_train["Task_ID"])
val_tasks   = set(ltr_val["Task_ID"])
test_tasks  = set(ltr_test["Task_ID"])

tv_overlap = train_tasks.intersection(val_tasks)
tt_overlap = train_tasks.intersection(test_tasks)
vt_overlap = val_tasks.intersection(test_tasks)

print("TASK LEAKAGE CHECK:")
print("=" * 55)
print(f"Train/Validation overlap:  {len(tv_overlap)}")
print(f"Train/Test overlap:        {len(tt_overlap)}")
print(f"Validation/Test overlap:   {len(vt_overlap)}")

assert train_tasks.isdisjoint(val_tasks),  "LEAKAGE: Train/Val!"
assert train_tasks.isdisjoint(test_tasks), "LEAKAGE: Train/Test!"
assert val_tasks.isdisjoint(test_tasks),   "LEAKAGE: Val/Test!"

print("PASS — No Task_ID leakage detected across any split!")

TASK LEAKAGE CHECK:
Train/Validation overlap:  0
Train/Test overlap:        0
Validation/Test overlap:   0
PASS — No Task_ID leakage detected across any split!


## Check E — Employee Cold-Start Coverage

In [6]:
train_employees = set(ltr_train["Employee_ID"])
val_employees   = set(ltr_val["Employee_ID"])
test_employees  = set(ltr_test["Employee_ID"])

val_missing  = val_employees  - train_employees
test_missing = test_employees - train_employees

print("EMPLOYEE COLD-START COVERAGE:")
print("=" * 55)
print(f"Train unique employees: {len(train_employees)}")
print(f"Val unique employees:   {len(val_employees)}")
print(f"Test unique employees:  {len(test_employees)}")
print()
print(f"Validation employees missing from training: {sorted(val_missing)}")
print(f"Test employees missing from training:       {sorted(test_missing)}")

if val_missing or test_missing:
    print()
    print("Cold-start employees detected.")
    print("Employee profile features for these employees are filled with 0.")
    print("The model cannot learn personalized history features for employees absent from training.")
else:
    print("PASS — All val/test employees appear in training data.")

EMPLOYEE COLD-START COVERAGE:
Train unique employees: 49
Val unique employees:   49
Test unique employees:  49

Validation employees missing from training: []
Test employees missing from training:       []
PASS — All val/test employees appear in training data.


## Check F — Ranking Group Arrays

In [7]:
print("RANKING GROUP ARRAY VERIFICATION:")
print("=" * 55)

ltr_train = ltr_train.sort_values("Task_ID").reset_index(drop=True)
ltr_val   = ltr_val.sort_values("Task_ID").reset_index(drop=True)
ltr_test  = ltr_test.sort_values("Task_ID").reset_index(drop=True)

train_group_sizes = ltr_train.groupby("Task_ID", sort=False).size().to_numpy()
val_group_sizes   = ltr_val.groupby("Task_ID",   sort=False).size().to_numpy()
test_group_sizes  = ltr_test.groupby("Task_ID",  sort=False).size().to_numpy()

assert train_group_sizes.sum() == len(ltr_train), "Train group sizes do not match row count!"
assert val_group_sizes.sum()   == len(ltr_val),   "Val group sizes do not match row count!"
assert test_group_sizes.sum()  == len(ltr_test),  "Test group sizes do not match row count!"

print(f"Train:  {len(train_group_sizes)} groups, sum={train_group_sizes.sum():,}, mean={train_group_sizes.mean():.1f}, min={train_group_sizes.min()}, max={train_group_sizes.max()}")
print(f"Val:    {len(val_group_sizes)} groups, sum={val_group_sizes.sum():,}, mean={val_group_sizes.mean():.1f}, min={val_group_sizes.min()}, max={val_group_sizes.max()}")
print(f"Test:   {len(test_group_sizes)} groups, sum={test_group_sizes.sum():,}, mean={test_group_sizes.mean():.1f}, min={test_group_sizes.min()}, max={test_group_sizes.max()}")
print("PASS — All group size assertions passed!")

LTR_DIR = "../data/processed/ltr_datasets"
with open(f"{LTR_DIR}/train_group_sizes.json",      "w") as f: json.dump(train_group_sizes.tolist(), f)
with open(f"{LTR_DIR}/validation_group_sizes.json", "w") as f: json.dump(val_group_sizes.tolist(), f)
with open(f"{LTR_DIR}/test_group_sizes.json",       "w") as f: json.dump(test_group_sizes.tolist(), f)
print("Group size arrays saved: train_group_sizes.json, validation_group_sizes.json, test_group_sizes.json")

RANKING GROUP ARRAY VERIFICATION:
Train:  1709 groups, sum=36,605, mean=21.4, min=21, max=49
Val:    366 groups, sum=7,829, mean=21.4, min=21, max=44
Test:   367 groups, sum=7,827, mean=21.3, min=21, max=34
PASS — All group size assertions passed!
Group size arrays saved: train_group_sizes.json, validation_group_sizes.json, test_group_sizes.json


## Final — Save Validation Report

In [8]:
all_features_numeric = all(
    pd.api.types.is_numeric_dtype(ltr_train[c])
    for c in FEATURE_COLS if c in ltr_train.columns
)
missing_feature_cols = [c for c in FEATURE_COLS if c not in ltr_train.columns]

validation_report = {
    "created_at": datetime.datetime.now().isoformat(),
    "dataset_row_counts": {
        "train": int(len(ltr_train)),
        "val":   int(len(ltr_val)),
        "test":  int(len(ltr_test)),
    },
    "unique_task_counts": {
        "train": int(ltr_train["Task_ID"].nunique()),
        "val":   int(ltr_val["Task_ID"].nunique()),
        "test":  int(ltr_test["Task_ID"].nunique()),
    },
    "unique_employee_counts": {
        "train": int(len(train_employees)),
        "val":   int(len(val_employees)),
        "test":  int(len(test_employees)),
    },
    "positive_counts": {
        "train": int((ltr_train["relevance"]==1).sum()),
        "val":   int((ltr_val["relevance"]==1).sum()),
        "test":  int((ltr_test["relevance"]==1).sum()),
    },
    "negative_counts": {
        "train": int((ltr_train["relevance"]==0).sum()),
        "val":   int((ltr_val["relevance"]==0).sum()),
        "test":  int((ltr_test["relevance"]==0).sum()),
    },
    "task_overlap_counts": {
        "train_val":  int(len(tv_overlap)),
        "train_test": int(len(tt_overlap)),
        "val_test":   int(len(vt_overlap)),
    },
    "cold_start_employee_counts": {
        "val":  int(len(val_missing)),
        "test": int(len(test_missing)),
    },
    "cold_start_employees": {
        "val":  sorted(val_missing),
        "test": sorted(test_missing),
    },
    "group_size_stats": {
        "train": {"count": int(len(train_group_sizes)), "mean": float(train_group_sizes.mean()), "min": int(train_group_sizes.min()), "max": int(train_group_sizes.max())},
        "val":   {"count": int(len(val_group_sizes)),   "mean": float(val_group_sizes.mean()),   "min": int(val_group_sizes.min()),   "max": int(val_group_sizes.max())},
        "test":  {"count": int(len(test_group_sizes)),  "mean": float(test_group_sizes.mean()),  "min": int(test_group_sizes.min()),  "max": int(test_group_sizes.max())},
    },
    "feature_col_count": len(FEATURE_COLS),
    "missing_feature_cols": missing_feature_cols,
    "all_features_numeric": bool(all_features_numeric),
    "checks": {
        "columns_present": len(missing_cols) == 0,
        "task_leakage_zero": len(tv_overlap) == 0 and len(tt_overlap) == 0 and len(vt_overlap) == 0,
        "group_sizes_valid": True,
        "all_features_numeric": bool(all_features_numeric),
    }
}

LTR_DIR = "../data/processed/ltr_datasets"
with open(f"{LTR_DIR}/ltr_validation_report.json", "w") as f:
    json.dump(validation_report, f, indent=2)

print("=" * 60)
print("NOTEBOOK 06 VALIDATION SUMMARY")
print("=" * 60)
print(f"Train: {len(ltr_train):,} rows, {ltr_train['Task_ID'].nunique()} tasks")
print(f"Val:   {len(ltr_val):,} rows, {ltr_val['Task_ID'].nunique()} tasks")
print(f"Test:  {len(ltr_test):,} rows, {ltr_test['Task_ID'].nunique()} tasks")
print(f"Feature columns: {len(FEATURE_COLS)}")
print(f"Missing required cols: {missing_cols}")
print(f"Task leakage: {len(tv_overlap)} / {len(tt_overlap)} / {len(vt_overlap)}")
print(f"Cold-start employees: val={len(val_missing)}, test={len(test_missing)}")
print(f"All features numeric: {all_features_numeric}")
print(f"ltr_validation_report.json saved.")
print("=" * 60)
print("Ready for model training in 07_Model_Training.ipynb")

NOTEBOOK 06 VALIDATION SUMMARY
Train: 36,605 rows, 1709 tasks
Val:   7,829 rows, 366 tasks
Test:  7,827 rows, 367 tasks
Feature columns: 56
Missing required cols: []
Task leakage: 0 / 0 / 0
Cold-start employees: val=0, test=0
All features numeric: True
ltr_validation_report.json saved.
Ready for model training in 07_Model_Training.ipynb
